In [27]:
# ============================================================
# PHARMACHAIN - COMPLETE BLOCKCHAIN CODE
# ============================================================

import json
from pathlib import Path
from datetime import datetime
from web3 import Web3


# ============================================================
# 1. BLOCKCHAIN CONNECTION
# ============================================================

RPC_URL = "http://127.0.0.1:8545"

w3 = Web3(
    Web3.HTTPProvider(RPC_URL)
)

if not w3.is_connected():
    raise Exception(
        "Cannot connect to Hardhat. "
        "Make sure 'npx hardhat node' is running."
    )

print("==============================================")
print("        PHARMACHAIN BLOCKCHAIN SYSTEM")
print("==============================================")
print("Blockchain Connected : True")
print("Chain ID             :", w3.eth.chain_id)


# ============================================================
# 2. LOAD CONTRACT ARTIFACT
# ============================================================

ARTIFACT_PATH = Path(
    r"C:\Users\HP\Desktop\BlockChain_Project"
    r"\artifacts\contracts\DrugTracking.sol"
    r"\DrugTracking.json"
)

if not ARTIFACT_PATH.exists():
    raise Exception(
        "DrugTracking.json not found.\n"
        "Run 'npx hardhat compile' first."
    )

with open(ARTIFACT_PATH, "r") as file:
    artifact = json.load(file)

ABI = artifact["abi"]

print("ABI Loaded          : True")


# ============================================================
# 3. LOAD CONTRACT ADDRESS
# ============================================================

ADDRESS_FILE = Path(
    r"C:\Users\HP\Desktop\BlockChain_Project"
    r"\Blockchain\contract_address.txt"
)

if not ADDRESS_FILE.exists():
    raise Exception(
        "contract_address.txt not found.\n"
        "Deploy the smart contract first."
    )

with open(ADDRESS_FILE, "r") as file:
    CONTRACT_ADDRESS = file.read().strip()

if not CONTRACT_ADDRESS:
    raise Exception(
        "Contract address file is empty."
    )

CONTRACT_ADDRESS = Web3.to_checksum_address(
    CONTRACT_ADDRESS
)

print("Contract Address    :", CONTRACT_ADDRESS)


# ============================================================
# 4. CONNECT TO SMART CONTRACT
# ============================================================

contract = w3.eth.contract(
    address=CONTRACT_ADDRESS,
    abi=ABI
)

print("Smart Contract      : Connected")


# ============================================================
# 5. BLOCKCHAIN ACCOUNTS / ROLES
# ============================================================

accounts = w3.eth.accounts

if len(accounts) < 3:
    raise Exception(
        "At least 3 Hardhat accounts are required."
    )

manufacturer = accounts[0]
distributor = accounts[1]
hospital = accounts[2]

print()
print("========== BLOCKCHAIN ROLES ==========")
print("Manufacturer :", manufacturer)
print("Distributor  :", distributor)
print("Hospital     :", hospital)
print("=======================================")


# ============================================================
# 6. STATUS DEFINITIONS
# ============================================================

STATUS_NAMES = {
    0: "MANUFACTURED",
    1: "SHIPPED",
    2: "RECEIVED",
    3: "DISPENSED"
}


def get_status_name(status):

    return STATUS_NAMES.get(
        int(status),
        "UNKNOWN"
    )


# ============================================================
# 7. BLOCKCHAIN CONNECTION FUNCTION
# ============================================================

def blockchain_connected():

    return w3.is_connected()


# ============================================================
# 8. CHECK DRUG EXISTENCE
# ============================================================

def blockchain_drug_exists(drug_id):

    return contract.functions.drugExists(
        drug_id
    ).call()


# ============================================================
# 9. MANUFACTURE DRUG
# ============================================================

def manufacture_drug(
    drug_id,
    batch_id,
    drug_name,
    manufacturer_name,
    quantity,
    manufacturing_date,
    expiry_date,
    sender
):

    sender = Web3.to_checksum_address(
        sender
    )

    transaction = contract.functions.manufactureDrug(
        drug_id,
        batch_id,
        drug_name,
        manufacturer_name,
        quantity,
        manufacturing_date,
        expiry_date
    ).transact({
        "from": sender
    })

    receipt = w3.eth.wait_for_transaction_receipt(
        transaction
    )

    return {
        "action": "MANUFACTURE",
        "transaction_hash":
            receipt["transactionHash"].hex(),
        "block_number":
            receipt["blockNumber"],
        "sender":
            sender
    }


# ============================================================
# 10. GET DRUG
# ============================================================

def get_drug(drug_id):

    return contract.functions.getDrug(
        drug_id
    ).call()


# ============================================================
# 11. SHIP DRUG
# Manufacturer → Distributor
# ============================================================

def ship_drug(
    drug_id,
    receiver_address,
    sender_address
):

    sender_address = Web3.to_checksum_address(
        sender_address
    )

    receiver_address = Web3.to_checksum_address(
        receiver_address
    )

    transaction = contract.functions.shipDrug(
        drug_id,
        receiver_address
    ).transact({
        "from": sender_address
    })

    receipt = w3.eth.wait_for_transaction_receipt(
        transaction
    )

    return {
        "action": "SHIP",
        "transaction_hash":
            receipt["transactionHash"].hex(),
        "block_number":
            receipt["blockNumber"],
        "sender":
            sender_address,
        "receiver":
            receiver_address
    }


# ============================================================
# 12. RECEIVE DRUG
# Distributor / Hospital
# ============================================================

def receive_drug(
    drug_id,
    receiver_address
):

    receiver_address = Web3.to_checksum_address(
        receiver_address
    )

    transaction = contract.functions.receiveDrug(
        drug_id
    ).transact({
        "from": receiver_address
    })

    receipt = w3.eth.wait_for_transaction_receipt(
        transaction
    )

    return {
        "action": "RECEIVE",
        "transaction_hash":
            receipt["transactionHash"].hex(),
        "block_number":
            receipt["blockNumber"],
        "receiver":
            receiver_address
    }


# ============================================================
# 13. SHIP TO HOSPITAL
# Distributor → Hospital
# ============================================================

def ship_to_hospital(
    drug_id,
    hospital_address,
    distributor_address
):

    distributor_address = Web3.to_checksum_address(
        distributor_address
    )

    hospital_address = Web3.to_checksum_address(
        hospital_address
    )

    transaction = contract.functions.shipToHospital(
        drug_id,
        hospital_address
    ).transact({
        "from": distributor_address
    })

    receipt = w3.eth.wait_for_transaction_receipt(
        transaction
    )

    return {
        "action": "SHIP_TO_HOSPITAL",
        "transaction_hash":
            receipt["transactionHash"].hex(),
        "block_number":
            receipt["blockNumber"],
        "sender":
            distributor_address,
        "receiver":
            hospital_address
    }


# ============================================================
# 14. DISPENSE DRUG
# Hospital → Patient
# ============================================================

def dispense_drug(
    drug_id,
    hospital_address
):

    hospital_address = Web3.to_checksum_address(
        hospital_address
    )

    transaction = contract.functions.dispenseDrug(
        drug_id
    ).transact({
        "from": hospital_address
    })

    receipt = w3.eth.wait_for_transaction_receipt(
        transaction
    )

    return {
        "action": "DISPENSE",
        "transaction_hash":
            receipt["transactionHash"].hex(),
        "block_number":
            receipt["blockNumber"],
        "sender":
            hospital_address
    }


# ============================================================
# 15. DISPLAY DRUG INFORMATION
# ============================================================

def display_drug(drug_id):

    drug = get_drug(drug_id)

    print()
    print("=" * 65)
    print("                    DRUG RECORD")
    print("=" * 65)

    print("Drug ID              :", drug[0])
    print("Batch ID             :", drug[1])
    print("Drug Name            :", drug[2])
    print("Manufacturer Name    :", drug[3])
    print("Quantity             :", drug[4])

    print(
        "Manufacturing Date   :",
        datetime.fromtimestamp(
            drug[5]
        )
    )

    print(
        "Expiry Date          :",
        datetime.fromtimestamp(
            drug[6]
        )
    )

    if drug[7] != 0:

        print(
            "Shipping Date        :",
            datetime.fromtimestamp(
                drug[7]
            )
        )

    else:

        print(
            "Shipping Date        : Not shipped"
        )

    if drug[8] != 0:

        print(
            "Receiving Date       :",
            datetime.fromtimestamp(
                drug[8]
            )
        )

    else:

        print(
            "Receiving Date       : Not received"
        )

    print(
        "Manufacturer Wallet  :",
        drug[9]
    )

    print(
        "Current Owner        :",
        drug[10]
    )

    print(
        "Status               :",
        get_status_name(
            drug[11]
        )
    )

    print(
        "Exists               :",
        drug[12]
    )

    print("=" * 65)

    return drug


# ============================================================
# 16. SYSTEM SUMMARY
# ============================================================

print()
print("==============================================")
print("        BLOCKCHAIN SYSTEM READY")
print("==============================================")
print("Functions Available:")
print()
print("1. blockchain_connected()")
print("2. blockchain_drug_exists()")
print("3. manufacture_drug()")
print("4. get_drug()")
print("5. ship_drug()")
print("6. receive_drug()")
print("7. ship_to_hospital()")
print("8. dispense_drug()")
print("9. display_drug()")
print()
print("Status:")
print("0 → MANUFACTURED")
print("1 → SHIPPED")
print("2 → RECEIVED")
print("3 → DISPENSED")
print("==============================================")

        PHARMACHAIN BLOCKCHAIN SYSTEM
Blockchain Connected : True
Chain ID             : 31337
ABI Loaded          : True
Contract Address    : 0x5FbDB2315678afecb367f032d93F642f64180aa3
Smart Contract      : Connected

========== BLOCKCHAIN ROLES ==========
Manufacturer : 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Distributor  : 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Hospital     : 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC

        BLOCKCHAIN SYSTEM READY
Functions Available:

1. blockchain_connected()
2. blockchain_drug_exists()
3. manufacture_drug()
4. get_drug()
5. ship_drug()
6. receive_drug()
7. ship_to_hospital()
8. dispense_drug()
9. display_drug()

Status:
0 → MANUFACTURED
1 → SHIPPED
2 → RECEIVED
3 → DISPENSED


In [29]:
#test drug
drug_id = "DRG-2026-TEST"
batch_id = "BATCH-001"
drug_name = "Paracetamol"
manufacturer_name = "ABC Pharma"
quantity = 1000

manufacturing_date = int(
    datetime(2026, 9, 8).timestamp()
)

expiry_date = int(
    datetime(2028, 9, 8).timestamp()
)

In [30]:
# Manufacture
print(
    manufacture_drug(
        drug_id,
        batch_id,
        drug_name,
        manufacturer_name,
        quantity,
        manufacturing_date,
        expiry_date,
        manufacturer
    )
)

# Manufacturer → Distributor
print(
    ship_drug(
        drug_id,
        distributor,
        manufacturer
    )
)

# Distributor receives
print(
    receive_drug(
        drug_id,
        distributor
    )
)

# Distributor → Hospital
print(
    ship_to_hospital(
        drug_id,
        hospital,
        distributor
    )
)

# Hospital receives
print(
    receive_drug(
        drug_id,
        hospital
    )
)

# Hospital → Patient
print(
    dispense_drug(
        drug_id,
        hospital
    )
)

{'action': 'MANUFACTURE', 'transaction_hash': 'fac17d192164a6efebafaf5d78e32ea0c27c83451300476dac98e78a6eb94bc3', 'block_number': 2, 'sender': '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266'}
{'action': 'SHIP', 'transaction_hash': 'dca8a9dad0ce03b8507e13e6207f83da77399e5322118974b077574db2b2f3a5', 'block_number': 3, 'sender': '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266', 'receiver': '0x70997970C51812dc3A010C7d01b50e0d17dc79C8'}
{'action': 'RECEIVE', 'transaction_hash': '730e8b4f9892d8f862c11a0dab8a2bcbbd55cf6dc56c8f040f6475823744bac7', 'block_number': 4, 'receiver': '0x70997970C51812dc3A010C7d01b50e0d17dc79C8'}
{'action': 'SHIP_TO_HOSPITAL', 'transaction_hash': '24ac01a0747a587cf8bc402c31ba7a76ccb1b8f1878adb6161076005c69c8500', 'block_number': 5, 'sender': '0x70997970C51812dc3A010C7d01b50e0d17dc79C8', 'receiver': '0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC'}
{'action': 'RECEIVE', 'transaction_hash': '87c7fb316e0ae2e1710c320fe472004fdb6537ea7d47c0b639a2134289b34ec4', 'block_number': 6, 're

In [31]:
display_drug(drug_id)


                    DRUG RECORD
Drug ID              : DRG-2026-TEST
Batch ID             : BATCH-001
Drug Name            : Paracetamol
Manufacturer Name    : ABC Pharma
Quantity             : 1000
Manufacturing Date   : 2026-09-08 00:00:00
Expiry Date          : 2028-09-08 00:00:00
Shipping Date        : 2026-09-12 14:41:42
Receiving Date       : 2026-09-12 14:41:43
Manufacturer Wallet  : 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Current Owner        : 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC
Status               : DISPENSED
Exists               : True


('DRG-2026-TEST',
 'BATCH-001',
 'Paracetamol',
 'ABC Pharma',
 1000,
 1788805800,
 1851964200,
 1789204302,
 1789204303,
 '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266',
 '0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC',
 3,
 True)

In [10]:
result = manufacture_drug(
    drug_id,
    batch_id,
    drug_name,
    manufacturer_name,
    quantity,
    manufacturing_date,
    expiry_date
)

print(result)

{'transaction_hash': '73a083b0c2b8a5efd7301c5bf8c28c4f3660f6db09c8eb8ef03b48dea13be978', 'block_number': 2, 'sender': '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266'}


In [11]:
exists = blockchain_drug_exists(drug_id)

print("Drug exists:", exists)

Drug exists: True


In [12]:
drug = get_drug(drug_id)

print(drug)

('DRG-2026-TEST', 'BATCH-001', 'Paracetamol', 'ABC Pharma', 1000, 1788805800, 1851964200, 0, 0, '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266', '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266', 0, True)


In [13]:
print("========== DRUG RECORD ==========")

print("Drug ID:", drug[0])
print("Batch ID:", drug[1])
print("Drug Name:", drug[2])
print("Manufacturer Name:", drug[3])
print("Quantity:", drug[4])

print("Manufacturing Timestamp:", drug[5])
print("Expiry Timestamp:", drug[6])
print("Shipping Timestamp:", drug[7])
print("Receiving Timestamp:", drug[8])

print("Manufacturer Wallet:", drug[9])
print("Current Owner:", drug[10])
print("Status:", drug[11])
print("Exists:", drug[12])

print("=================================")

========== DRUG RECORD ==========
Drug ID: DRG-2026-TEST
Batch ID: BATCH-001
Drug Name: Paracetamol
Manufacturer Name: ABC Pharma
Quantity: 1000
Manufacturing Timestamp: 1788805800
Expiry Timestamp: 1851964200
Shipping Timestamp: 0
Receiving Timestamp: 0
Manufacturer Wallet: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Current Owner: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Status: 0
Exists: True


In [35]:
# ============================================================
# PHARMACHAIN - BLOCKCHAIN + SQLITE INTEGRATION
# ============================================================

import importlib.util
from pathlib import Path


# ============================================================
# 1. LOAD database.py DIRECTLY
# ============================================================

DATABASE_PY = Path(
    r"C:\Users\HP\Desktop\BlockChain_Project"
    r"\database\database.py"
)

if not DATABASE_PY.exists():
    raise FileNotFoundError(
        f"database.py not found at:\n{DATABASE_PY}"
    )

spec = importlib.util.spec_from_file_location(
    "pharmachain_database",
    DATABASE_PY
)

db = importlib.util.module_from_spec(spec)
spec.loader.exec_module(db)

print("SQLite module loaded successfully.")


# ============================================================
# 2. CHECK EXISTING DATABASE
# ============================================================

DATABASE_FILE = Path(
    r"C:\Users\HP\Desktop\BlockChain_Project"
    r"\database\drugs.db"
)

if not DATABASE_FILE.exists():
    raise FileNotFoundError(
        f"drugs.db not found at:\n{DATABASE_FILE}"
    )

print("SQLite database found.")
print("Database:", DATABASE_FILE)


# ============================================================
# 3. MANUFACTURE + DATABASE
# ============================================================

def manufacture_drug_integrated(
    drug_id,
    batch_id,
    drug_name,
    manufacturer_name,
    quantity,
    manufacturing_date,
    expiry_date
):

    # ---- BLOCKCHAIN ----
    blockchain_result = manufacture_drug(
        drug_id,
        batch_id,
        drug_name,
        manufacturer_name,
        quantity,
        manufacturing_date,
        expiry_date,
        manufacturer
    )

    # ---- SQLITE ----
    db.add_drug(
        drug_id=drug_id,
        drug_name=drug_name,
        batch_number=batch_id,
        manufacturing_date=str(
            manufacturing_date
        ),
        expiry_date=str(
            expiry_date
        ),
        quantity=quantity,
        manufacturer=manufacturer,
        qr_data=drug_id
    )

    db.add_transaction(
        drug_id=drug_id,
        action="MANUFACTURE",
        sender=manufacturer,
        receiver=manufacturer,
        block_number=blockchain_result["block_number"],
        transaction_hash=blockchain_result[
            "transaction_hash"
        ]
    )

    return blockchain_result


# ============================================================
# 4. MANUFACTURER → DISTRIBUTOR
# ============================================================

def ship_drug_integrated(drug_id):

    # ---- BLOCKCHAIN ----
    blockchain_result = ship_drug(
        drug_id,
        distributor,
        manufacturer
    )

    # ---- SQLITE ----
    db.update_drug(
        drug_id,
        current_owner=distributor,
        status="SHIPPED"
    )

    db.add_transaction(
        drug_id=drug_id,
        action="SHIP",
        sender=manufacturer,
        receiver=distributor,
        block_number=blockchain_result["block_number"],
        transaction_hash=blockchain_result[
            "transaction_hash"
        ]
    )

    return blockchain_result


# ============================================================
# 5. DISTRIBUTOR RECEIVES
# ============================================================

def distributor_receive_integrated(drug_id):

    # ---- BLOCKCHAIN ----
    blockchain_result = receive_drug(
        drug_id,
        distributor
    )

    # ---- SQLITE ----
    db.update_drug(
        drug_id,
        current_owner=distributor,
        status="RECEIVED"
    )

    db.add_transaction(
        drug_id=drug_id,
        action="RECEIVE",
        sender=distributor,
        receiver=distributor,
        block_number=blockchain_result["block_number"],
        transaction_hash=blockchain_result[
            "transaction_hash"
        ]
    )

    return blockchain_result


# ============================================================
# 6. DISTRIBUTOR → HOSPITAL
# ============================================================

def ship_to_hospital_integrated(drug_id):

    # ---- BLOCKCHAIN ----
    blockchain_result = ship_to_hospital(
        drug_id,
        hospital,
        distributor
    )

    # ---- SQLITE ----
    db.update_drug(
        drug_id,
        current_owner=hospital,
        status="SHIPPED"
    )

    db.add_transaction(
        drug_id=drug_id,
        action="SHIP_TO_HOSPITAL",
        sender=distributor,
        receiver=hospital,
        block_number=blockchain_result["block_number"],
        transaction_hash=blockchain_result[
            "transaction_hash"
        ]
    )

    return blockchain_result


# ============================================================
# 7. HOSPITAL RECEIVES
# ============================================================

def hospital_receive_integrated(drug_id):

    # ---- BLOCKCHAIN ----
    blockchain_result = receive_drug(
        drug_id,
        hospital
    )

    # ---- SQLITE ----
    db.update_drug(
        drug_id,
        current_owner=hospital,
        status="RECEIVED"
    )

    db.add_transaction(
        drug_id=drug_id,
        action="RECEIVE",
        sender=hospital,
        receiver=hospital,
        block_number=blockchain_result["block_number"],
        transaction_hash=blockchain_result[
            "transaction_hash"
        ]
    )

    return blockchain_result


# ============================================================
# 8. HOSPITAL → PATIENT
# ============================================================

def dispense_drug_integrated(drug_id):

    # ---- BLOCKCHAIN ----
    blockchain_result = dispense_drug(
        drug_id,
        hospital
    )

    # ---- SQLITE ----
    db.update_drug(
        drug_id,
        current_owner=hospital,
        status="DISPENSED"
    )

    db.add_transaction(
        drug_id=drug_id,
        action="DISPENSE",
        sender=hospital,
        receiver="PATIENT",
        block_number=blockchain_result["block_number"],
        transaction_hash=blockchain_result[
            "transaction_hash"
        ]
    )

    return blockchain_result


# ============================================================
# 9. DISPLAY COMPLETE DATABASE HISTORY
# ============================================================

def display_database_history(drug_id):

    drug = db.get_drug(drug_id)
    transactions = db.get_transactions(drug_id)

    print()
    print("=" * 80)
    print("                 SQLITE DRUG RECORD")
    print("=" * 80)

    if drug is None:
        print("Drug not found in SQLite database.")
        return

    print("Drug ID          :", drug[0])
    print("Drug Name        :", drug[1])
    print("Batch Number     :", drug[2])
    print("Manufacturing    :", drug[3])
    print("Expiry           :", drug[4])
    print("Quantity         :", drug[5])
    print("Manufacturer     :", drug[6])
    print("Current Owner    :", drug[7])
    print("Status           :", drug[8])
    print("QR Data          :", drug[9])
    print("Created At       :", drug[10])

    print()
    print("=" * 80)
    print("                 TRANSACTION HISTORY")
    print("=" * 80)

    for tx in transactions:

        print()
        print("Transaction ID   :", tx[0])
        print("Action           :", tx[1])
        print("Sender           :", tx[2])
        print("Receiver         :", tx[3])
        print("Timestamp        :", tx[4])
        print("Block Number     :", tx[5])
        print("Transaction Hash :", tx[6])

    print()
    print("=" * 80)


# ============================================================
# 10. INTEGRATION READY
# ============================================================

print()
print("==============================================")
print("   PHARMACHAIN BLOCKCHAIN + SQLITE READY")
print("==============================================")
print()
print("Integrated functions:")
print()
print("1. manufacture_drug_integrated()")
print("2. ship_drug_integrated()")
print("3. distributor_receive_integrated()")
print("4. ship_to_hospital_integrated()")
print("5. hospital_receive_integrated()")
print("6. dispense_drug_integrated()")
print("7. display_database_history()")
print()
print("Blockchain + SQLite connection successful.")
print("==============================================")

SQLite module loaded successfully.
SQLite database found.
Database: C:\Users\HP\Desktop\BlockChain_Project\database\drugs.db

   PHARMACHAIN BLOCKCHAIN + SQLITE READY

Integrated functions:

1. manufacture_drug_integrated()
2. ship_drug_integrated()
3. distributor_receive_integrated()
4. ship_to_hospital_integrated()
5. hospital_receive_integrated()
6. dispense_drug_integrated()
7. display_database_history()

Blockchain + SQLite connection successful.


In [ ]:
#